# Emoji Semantic Search
THA-1 (Word Representation), Question 1: Creative Semantic Retrieval

The idea is to type a feeling or situation in plain English, for example "celebrating a big win", and get back the 5 emojis that match it best.

Steps:
1. Build a dataset of emojis, each with a short text description (its Unicode name and keywords).
2. Turn every description into a vector using a pretrained sentence embedding model.
3. Turn the query into a vector with the same model and compare it with every emoji using cosine similarity.
4. Sort by similarity and return the top 5.
5. Test 10 queries, judge the results by hand, and discuss where it works and where it doesn't.

## 1. Setup

In [18]:
%pip install -q sentence-transformers

In [19]:
import os, json, urllib.request, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 120)

## 2. Dataset
The emoji data comes from two official Unicode files:
- [CLDR English annotations](https://github.com/unicode-org/cldr-json/blob/main/cldr-json/cldr-annotations-full/annotations/en/annotations.json), which give each emoji a short name and a list of keywords.
- [emoji-test.txt](https://unicode.org/Public/emoji/latest/emoji-test.txt) (Emoji 18.0), the official list of emojis. I used it to filter out plain symbols like `{` that are also in the CLDR file.

Each emoji's text is written as `"name: keyword, keyword, ..."`, and this is the only thing the model sees.

I removed skin-tone variants (2,040), flags (262), emojis with no CLDR keywords (73), and man/woman versions of emojis that already have a gender-neutral version (130), since these would mostly fill the results with duplicates. That leaves 1,458 emojis.

The dataset is saved as `emojis.csv`, so the download only runs if that file is missing.

In [20]:
CLDR_URL = "https://raw.githubusercontent.com/unicode-org/cldr-json/main/cldr-json/cldr-annotations-full/annotations/en/annotations.json"
EMOJI_TEST_URL = "https://unicode.org/Public/emoji/latest/emoji-test.txt"
SKIN_TONES = {chr(c) for c in range(0x1F3FB, 0x1F400)}

def fetch(url):
    with urllib.request.urlopen(url) as r:
        return r.read().decode("utf-8")

def build_emoji_dataset(path="emojis.csv"):
    annotations = json.loads(fetch(CLDR_URL))["annotations"]["annotations"]  # {emoji: {"tts": [name], "default": [keywords]}}
    strip_vs = lambda s: s.replace("\uFE0F", "")                            # CLDR keys omit the invisible variation selector
    rows, skipped = [], {"skin tone": 0, "flag": 0, "no CLDR keywords": 0}
    for line in fetch(EMOJI_TEST_URL).splitlines():
        if "; fully-qualified" not in line:
            continue
        emo, _version, *name_words = line.split("# ", 1)[1].split()          # "... # 😂 E0.6 face with tears of joy"
        if SKIN_TONES & set(emo):
            skipped["skin tone"] += 1
        elif " ".join(name_words).startswith("flag:"):
            skipped["flag"] += 1
        elif strip_vs(emo) not in annotations:
            skipped["no CLDR keywords"] += 1
        else:
            entry = annotations[strip_vs(emo)]
            name, keywords = entry["tts"][0], ", ".join(entry["default"])
            rows.append({"emoji": emo, "name": name, "keywords": keywords, "text": f"{name}: {keywords}"})
    data = pd.DataFrame(rows)
    names = set(data["name"])
    def is_gender_variant(name):                                             # "man facepalming" when "person facepalming" exists
        for prefix in ("man ", "woman "):
            if name.startswith(prefix):
                rest = name[len(prefix):]
                return rest in names or ("person " + rest) in names
        return False
    gendered = data["name"].apply(is_gender_variant)
    skipped["gendered variant"] = int(gendered.sum())
    data[~gendered].to_csv(path, index=False, encoding="utf-8")
    print("Excluded:", skipped)

REBUILD = False   # set to True to re-download from Unicode and rebuild emojis.csv
if REBUILD or not os.path.exists("emojis.csv"):
    build_emoji_dataset()

df = pd.read_csv("emojis.csv")
print("Number of emojis:", len(df))
df.sample(8, random_state=1)

Number of emojis: 1458


,emoji,name,keywords,text
1162,🪃,boomerang,"boomerang, rebound, repercussion, weapon","boomerang: boomerang, rebound, repercussion, weapon"
111,💩,pile of poo,"bs, comic, doo, dung, face, fml, monster, pile, poo, poop, smelly, smh, stink, stinks, stinky, turd","pile of poo: bs, comic, doo, dung, face, fml, monster, pile, poo, poop, smelly, smh, stink, stinks, stinky, turd"
1272,🔚,END arrow,"arrow, END","END arrow: arrow, END"
1146,🗄️,file cabinet,"cabinet, file, filing, paper","file cabinet: cabinet, file, filing, paper"
368,🐵,monkey face,"animal, banana, face, monkey","monkey face: animal, banana, face, monkey"
37,🤨,face with raised eyebrow,"disapproval, disbelief, distrust, emoji, eyebrow, face, hmm, mild, raised, skeptic, skeptical, skepticism, surprise,...","face with raised eyebrow: disapproval, disbelief, distrust, emoji, eyebrow, face, hmm, mild, raised, skeptic, skepti..."
1127,📂,open file folder,"file, folder, open","open file folder: file, folder, open"
993,🥿,flat shoe,"ballet, comfy, flat, flats, shoe, slip-on, slipper","flat shoe: ballet, comfy, flat, flats, shoe, slip-on, slipper"


## 3. Embeddings
I use the pretrained model [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) from Sentence Transformers. It turns a piece of text into a vector of 384 numbers, and texts with similar meanings end up with similar vectors. All emoji texts are embedded once and stored as a matrix with one row per emoji.

In [21]:
model = SentenceTransformer("all-MiniLM-L6-v2")
emoji_embeddings = model.encode(df["text"].tolist(), convert_to_numpy=True, show_progress_bar=False)

print("Embedding matrix shape:", emoji_embeddings.shape, "-> (number of emojis, numbers per emoji)")
print("First 8 of 384 numbers for", df.emoji[0], df.name[0], ":", np.round(emoji_embeddings[0, :8], 3))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding matrix shape: (1458, 384) -> (number of emojis, numbers per emoji)
First 8 of 384 numbers for 😀 grinning face : [-0.027  0.031 -0.011  0.035 -0.031 -0.009  0.127 -0.009]


## 4. Cosine similarity
$$\text{cosine}(a, b) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$$

Cosine similarity compares the direction of two vectors: 1 means they point the same way, 0 means they are unrelated. I wrote it with numpy so that one query is compared with every emoji at once.

In [22]:
def cosine_similarity(query_vec, matrix):
    dots = matrix @ query_vec                                           # dot product of the query with every row -> shape (n,)
    norms = np.linalg.norm(matrix, axis=1) * np.linalg.norm(query_vec)  # length of each row times length of the query
    return dots / norms                                                 # cosine = dot / (|a| * |b|)

A quick check on small vectors I can verify by hand, and on a few emoji pairs:

In [23]:
toy = np.array([[1, 0], [1, 1], [0, 1], [-1, 0], [5, 0]], dtype=float)
print("[1,0] vs [1,0], [1,1], [0,1], [-1,0], [5,0] ->", np.round(cosine_similarity(np.array([1.0, 0.0]), toy), 3))

row = lambda e: df.index[df.emoji == e][0]
for a, b in [("😂", "🤣"), ("😂", "😭"), ("😀", "😢"), ("🔥", "🧯"), ("😂", "🥦")]:
    s = cosine_similarity(emoji_embeddings[row(a)], emoji_embeddings[[row(b)]])[0]
    print(f"{a} {df.name[row(a)]:<24} vs {b} {df.name[row(b)]:<28} {s:.3f}")

[1,0] vs [1,0], [1,1], [0,1], [-1,0], [5,0] -> [ 1.     0.707  0.    -1.     1.   ]
😂 face with tears of joy   vs 🤣 rolling on the floor laughing 0.755
😂 face with tears of joy   vs 😭 loudly crying face           0.684
😀 grinning face            vs 😢 crying face                  0.444
🔥 fire                     vs 🧯 fire extinguisher            0.564
😂 face with tears of joy   vs 🥦 broccoli                     0.168


## 5. Retrieval
`retrieve` embeds the query with the same model, scores every emoji, and uses `np.argsort` to pick the top k. `rank_of` finds where a specific emoji ended up in the full ranking, which I use later to check emojis I expected but that didn't make the top 5.

In [24]:
def retrieve(query, k=5, matrix=emoji_embeddings):
    query_vec = model.encode(query, convert_to_numpy=True)       # query -> 384-number vector
    scores = cosine_similarity(query_vec, matrix)                 # one score per emoji
    top_idx = np.argsort(scores)[::-1][:k]                        # positions of the k highest scores
    results = df.loc[top_idx, ["emoji", "name"]].reset_index(drop=True)
    results.insert(0, "rank", range(1, k + 1))
    results["score"] = scores[top_idx].round(3)
    return results

def rank_of(query, emoji, matrix=emoji_embeddings):
    scores = cosine_similarity(model.encode(query, convert_to_numpy=True), matrix)
    order = np.argsort(scores)[::-1]                              # all emojis, best first
    position = np.where(order == row(emoji))[0][0] + 1            # 1-based rank of that emoji
    return int(position), round(float(scores[row(emoji)]), 3)

retrieve("pizza night with friends")

,rank,emoji,name,score
0,1,🍕,pizza,0.496
1,2,🌃,night with stars,0.426
2,3,🍝,spaghetti,0.385
3,4,🍧,shaved ice,0.337
4,5,🛏️,bed,0.329


## 6. Test queries
I picked 10 queries of different types to see where the system does well and where it struggles: clear emotions, situations, a mixed emotion, slang (where people use an emoji differently from its official meaning, like 💀 for something funny), negation, and one abstract query.

In [25]:
test_queries = [
    ("so happy I could cry",           "clear emotion"),
    ("feeling sleepy and bored",       "clear emotion"),
    ("celebrating a big win",          "situation"),
    ("stuck in traffic",               "situation"),
    ("nervous before a job interview", "situation"),
    ("proud but exhausted",            "mixed emotion"),
    ("that's hilarious",               "slang / social meaning"),
    ("that's really cool",             "slang / social meaning"),
    ("I'm not happy at all",           "negation"),
    ("the calm before a storm",        "abstract"),
]

results = {}
for query, qtype in test_queries:
    results[query] = retrieve(query)
    print(f"\nQuery: \"{query}\"   [{qtype}]")
    display(results[query])


Query: "so happy I could cry"   [clear emotion]


,rank,emoji,name,score
0,1,🥹,face holding back tears,0.499
1,2,🥲,smiling face with tear,0.488
2,3,😭,loudly crying face,0.465
3,4,😢,crying face,0.462
4,5,😂,face with tears of joy,0.447



Query: "feeling sleepy and bored"   [clear emotion]


,rank,emoji,name,score
0,1,🫩,face with bags under eyes,0.567
1,2,😪,sleepy face,0.517
2,3,🥱,yawning face,0.505
3,4,😩,weary face,0.500
4,5,😫,tired face,0.499



Query: "celebrating a big win"   [situation]


...

,rank,emoji,name,score
0,1,🆒,COOL button,0.258
1,2,🔄,counterclockwise arrows button,0.231
2,3,⌛,hourglass done,0.229
3,4,⏳,hourglass not done,0.209
4,5,🔃,clockwise vertical arrows,0.206



Query: "I'm not happy at all"   [negation]


,rank,emoji,name,score
0,1,😞,disappointed face,0.381
1,2,😥,sad but relieved face,0.359
2,3,😀,grinning face,0.348
3,4,😔,pensive face,0.332
4,5,🙂,slightly smiling face,0.331



Query: "the calm before a storm"   [abstract]


,rank,emoji,name,score
0,1,🌬️,wind face,0.495
1,2,🌁,foggy,0.470
2,3,🌪️,tornado,0.466
3,4,😌,relieved face,0.464
4,5,🎐,wind chime,0.456


For some queries there is an emoji I expected to see (💀 and 🔥 for the slang queries, 😮‍💨 for exhausted, ⛈️ for the storm) or one that shouldn't be there (😀 for "not happy"). This table shows where they actually ranked out of 1,458.

In [10]:
expected = [("that's hilarious", "💀"), ("that's hilarious", "😂"), ("that's really cool", "🔥"),
            ("that's really cool", "😎"), ("I'm not happy at all", "😞"), ("I'm not happy at all", "😀"),
            ("proud but exhausted", "😮‍💨"), ("the calm before a storm", "⛈️")]
pd.DataFrame([(q, e, df.name[row(e)], *rank_of(q, e)) for q, e in expected],
             columns=["query", "emoji", "name", "rank (of 1458)", "score"])

,query,emoji,name,rank (of 1458),score
0,that's hilarious,💀,skull,980,0.041
1,that's hilarious,😂,face with tears of joy,7,0.213
2,that's really cool,🔥,fire,833,0.061
3,that's really cool,😎,smiling face with sunglasses,23,0.181
4,I'm not happy at all,😞,disappointed face,1,0.381
5,I'm not happy at all,😀,grinning face,3,0.348
6,proud but exhausted,😮‍💨,face exhaling,26,0.256
7,the calm before a storm,⛈️,cloud with lightning and rain,10,0.416


## 7. Manual assessment
I marked each result as relevant (1) or not relevant (0), based on whether someone typing the query would actually pick that emoji. Precision@5 is the number of relevant results divided by 5.

In [26]:
judgments = {
    "so happy I could cry": {"face holding back tears": 1, "smiling face with tear": 1,
                             "loudly crying face": 0,        # sad crying, not happy crying
                             "crying face": 0,               # sad
                             "face with tears of joy": 1},
    "feeling sleepy and bored": {"face with bags under eyes": 1, "sleepy face": 1, "yawning face": 1,
                                 "weary face": 1, "tired face": 1},   # all sleepy/tired, but none shows "bored"
    "celebrating a big win": {"ribbon": 0,                   # matched only the keyword "celebration"
                              "trophy": 1, "clapping hands": 1, "confetti ball": 1, "sports medal": 1},
    "stuck in traffic": {"motorway": 1, "vertical traffic light": 1, "horizontal traffic light": 1,
                         "bus stop": 0, "no entry": 0},      # topic only; no frustration emoji like 😤
    "nervous before a job interview": {"anxious face with sweat": 1, "worried face": 1, "fearful face": 1,
                                       "downcast face with sweat": 1, "grinning face with sweat": 0},
    "proud but exhausted": {"weary face": 1, "grinning face with sweat": 1, "face holding back tears": 1,
                            "face with bags under eyes": 1,
                            "disappointed face": 0},         # contradicts "proud"
    "that's hilarious": {"grinning squinting face": 1, "rolling on the floor laughing": 1,
                         "astonished face": 0, "upside-down face": 0, "cat with tears of joy": 1},
    "that's really cool": {"COOL button": 1,                 # literal word match, but people do use it
                           "counterclockwise arrows button": 0, "hourglass done": 0,
                           "hourglass not done": 0, "clockwise vertical arrows": 0},
    "I'm not happy at all": {"disappointed face": 1, "sad but relieved face": 1,
                             "grinning face": 0,             # the opposite: negation ignored
                             "pensive face": 1, "slightly smiling face": 0},
    "the calm before a storm": {"wind face": 0, "foggy": 0,
                                "tornado": 1,                # lenient: matches the "storm" half only
                                "relieved face": 1,          # lenient: matches the "calm" half only
                                "wind chime": 0},
}

In [27]:
rows = []
for query, qtype in test_queries:
    top5 = results[query]["name"].tolist()
    relevant = [judgments[query][name] for name in top5]
    marks = " ".join(("✅" if r else "❌") + e for r, e in zip(relevant, results[query]["emoji"]))
    rows.append((query, qtype, marks, sum(relevant) / 5))

p_at_5 = pd.DataFrame(rows, columns=["query", "type", "judged top 5", "Precision@5"])
display(p_at_5)
print("Average Precision@5:", round(p_at_5["Precision@5"].mean(), 2))
print(p_at_5.groupby("type", sort=False)["Precision@5"].mean().round(2).to_string())

,query,type,judged top 5,Precision@5
0,so happy I could cry,clear emotion,✅🥹 ✅🥲 ❌😭 ❌😢 ✅😂,0.6
1,feeling sleepy and bored,clear emotion,✅🫩 ✅😪 ✅🥱 ✅😩 ✅😫,1.0
2,celebrating a big win,situation,❌🎀 ✅🏆 ✅👏 ✅🎊 ✅🏅,0.8
3,stuck in traffic,situation,✅🛣️ ✅🚦 ✅🚥 ❌🚏 ❌⛔,0.6
4,nervous before a job interview,situation,✅😰 ✅😟 ✅😨 ✅😓 ❌😅,0.8
5,proud but exhausted,mixed emotion,✅😩 ✅😅 ✅🥹 ✅🫩 ❌😞,0.8
6,that's hilarious,slang / social meaning,✅😆 ✅🤣 ❌😲 ❌🙃 ✅😹,0.6
7,that's really cool,slang / social meaning,✅🆒 ❌🔄 ❌⌛ ❌⏳ ❌🔃,0.2
8,I'm not happy at all,negation,✅😞 ✅😥 ❌😀 ✅😔 ❌🙂,0.6
9,the calm before a storm,abstract,❌🌬️ ❌🌁 ✅🌪️ ✅😌 ❌🎐,0.4


Average Precision@5: 0.64
type
clear emotion             0.80
situation                 0.73
mixed emotion             0.80
slang / social meaning    0.40
negation                  0.60
abstract                  0.40


## 8. Extra: name only vs name + keywords
To see how much the keywords matter, I also embedded each emoji using only its name (e.g. "skull") and ran the same queries.

In [28]:
name_embeddings = model.encode(df["name"].tolist(), convert_to_numpy=True, show_progress_bar=False)

comparison = []
for q, _ in test_queries:
    name_only = retrieve(q, matrix=name_embeddings)
    comparison.append((q, " ".join(name_only["emoji"]), " ".join(results[q]["emoji"])))
pd.DataFrame(comparison, columns=["query", "top 5: name only", "top 5: name + keywords"])

,query,top 5: name only,top 5: name + keywords
0,so happy I could cry,😂 😥 😢 🥹 🥲,🥹 🥲 😭 😢 😂
1,feeling sleepy and bored,😪 😫 💫 🥱 🌁,🫩 😪 🥱 😩 😫
2,celebrating a big win,🏆 ✌️ 👏 😃 🏅,🎀 🏆 👏 🎊 🏅
3,stuck in traffic,🚏 🛣️ 🚕 🚥 🎫,🛣️ 🚦 🚥 🚏 ⛔
4,nervous before a job interview,😰 😨 😟 😱 😓,😰 😟 😨 😓 😅
5,proud but exhausted,😫 😩 😪 😅 😌,😩 😅 🥹 🫩 😞
6,that's hilarious,🤣 🕉️ 😺 👦 💤,😆 🤣 😲 🙃 😹
7,that's really cool,🔗 🆒 🖼️ ⏲️ 🧉,🆒 🔄 ⌛ ⏳ 🔃
8,I'm not happy at all,😥 ☺️ 😞 😇 🙂,😞 😥 😀 😔 🙂
9,the calm before a storm,🌪️ 🎐 🌬️ 🌧️ 🌀,🌬️ 🌁 🌪️ 😌 🎐


In [29]:
checks = [("that's hilarious", "💀"), ("that's hilarious", "😂"), ("that's really cool", "🔥"),
          ("so happy I could cry", "🥹"), ("celebrating a big win", "🏆")]
pd.DataFrame([(q, e, rank_of(q, e, name_embeddings)[0], rank_of(q, e)[0]) for q, e in checks],
             columns=["query", "emoji", "rank: name only", "rank: name + keywords"])

,query,emoji,rank: name only,rank: name + keywords
0,that's hilarious,💀,350,980
1,that's hilarious,😂,227,7
2,that's really cool,🔥,1298,833
3,so happy I could cry,🥹,4,1
4,celebrating a big win,🏆,1,2


## 9. Discussion

### 9.1 Which types of queries produced good or poor results?
- **Good: clear emotions and situations whose words overlap the CLDR keywords.** "nervous before a job interview" (P@5 1.0: 😰 😟 😨 😓 😅) and "feeling sleepy and bored" (P@5 1.0, highest top score 0.567). "celebrating a big win" reached 0.8 (🏆 👏 🎊 🏅).
- **Poor: slang and abstract queries.** "that's really cool" had P@5 0.2: after 🆒, the results were arrows and hourglasses (🔄 ⌛ ⏳ 🔃) with no link to the query. "the calm before a storm" had 0.4, and only with lenient judging.
- **The top score is a rough warning sign:** the two slang queries had the lowest top scores (0.289 and 0.258), while the good queries started at 0.44–0.57. But it is not reliable on its own: the negation query started at a normal-looking 0.381 and still put 😀 at #3.
- By type (suggested judgments): clear emotion / situation / mixed 0.8, negation 0.6, slang / abstract 0.4.

### 9.2 Did the system retrieve items related to the topic but not the intended meaning?
Yes, often:
- **"so happy I could cry" → 😭 😢 (sad crying).** The word *cry* dominates. The sanity check already showed 😂 vs 😭 at 0.684.
- **"celebrating a big win" → 🎀 ribbon at #1 (0.522)**, only because its text is literally "ribbon: celebration, ribbon". With **name-only** text, 🏆 was #1 and 🎀 did not appear.
- **"stuck in traffic" → 🛣️ 🚦 🚥 🚏 ⛔**: the *topic* (roads) but not the *feeling* (frustration).
- **"I'm not happy at all" → 😀 grinning face at #3**: its text contains "happy".
- **"the calm before a storm" → 🌪️ tornado**, the storm itself rather than the uneasy calm before it.

### 9.3 Which semantic relationships were captured well or poorly?
- **Captured well: synonyms and paraphrases** between the query and the keywords (nervous → anxious / worried / fearful; sleepy → tired / yawning), and **category membership** (a win → trophy, medal, confetti).
- **Captured poorly:**
  - **Negation**: "not happy" still retrieves happy faces.
  - **Composition of two feelings**: "proud but exhausted" is mostly exhausted, and "sleepy and bored" is only sleepy.
  - **Social / pragmatic meaning**: 💀 ("I'm dead", meaning very funny) ranked **980 of 1,458** for "that's hilarious", and 🔥 ranked **833** for "that's really cool".
  - **Metaphor**: "calm before a storm".
- **Opposites can look similar:** 😀 vs 😢 scored 0.444. Words used in similar contexts (faces, emotions) get similar vectors even when their meanings are opposite.

### 9.4 Limitations and failure cases of embeddings for this application
- **The model sees only the text description, not the image or real-world usage.** Emoji meaning is social and changes over time, and that is not in the official descriptions.
- **The item text matters a lot, and keywords are a mixed blessing**. Keywords moved 😂 from rank 227 to 7 for "that's hilarious", but pushed 💀 from 350 to 980 (its keywords stress *dead / death / monster*) and introduced the 🎀 error.
- **Uneven descriptions:** ❤️ has only 4 keywords and lost to 🤟 (which literally contains "love-you") in the warm-up query "I love you so much"; 🔚 has only two keywords.
- **One vector per query**: multi-part queries get squeezed together, and one part tends to win.
- **Scores are not calibrated:** unrelated pairs still score about 0.17 (😂 vs 🥦), and the best matches are often only 0.3–0.5, so there is no natural threshold for "nothing relevant found".
- **Evaluation limits:** 10 queries, one judge and Precision@5 ignores good emojis that ranked 6th or lower.

## 10. Resources
- Data: [Unicode CLDR English emoji annotations](https://github.com/unicode-org/cldr-json/blob/main/cldr-json/cldr-annotations-full/annotations/en/annotations.json) and [Unicode emoji-test.txt](https://unicode.org/Public/emoji/latest/emoji-test.txt) (Emoji 18.0).
- Model: [all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) from Sentence Transformers, used as-is without any training.
- Libraries: sentence-transformers, numpy, pandas.